In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import pickle
import numpy as np

# Load your existing model
model = pickle.load(open("fraud_model (4).pkl", "rb"))

# --- HARD-CODED MAPPINGS (Extracted from your CSV) ---
# These represent how LabelEncoder converted your data alphabetically
mappings = {
    "Transaction_Type": ['Bank Transfer', 'Bill Payment', 'Investment', 'Other', 'Purchase', 'Refund', 'Subscription'],
    "Payment_Gateway": ['Alpha Bank', 'Bank of Data', 'CReditPAY', 'Dummy Bank', 'Gamma Bank', 'Other', 'SamplePay', 'Sigma Bank', 'UPI Pay'],
    "Transaction_Status": ['Completed', 'Failed', 'Pending'],
    "Device_OS": ['Android', 'MacOS', 'Windows', 'iOS'],
    "Merchant_Category": ['Brand Vouchers and OTT', 'Donations and Devotion', 'Financial services and Taxes', 'Home delivery', 'Investment', 'More Services', 'Other', 'Purchases', 'Travel bookings', 'Utilities'],
    "Transaction_Channel": ['In-store', 'Mobile', 'Online'],
    "Transaction_State": ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
}

def predict(
    Merchant_ID, Transaction_Type, Payment_Gateway, Transaction_City,
    Transaction_State, Transaction_Status, Device_OS, Transaction_Frequency,
    Merchant_Category, Transaction_Channel, Transaction_Amount_Deviation,
    Days_Since_Last_Transaction, amount
):
    # Convert text inputs to numbers using our mapping
    # .index() finds the position of the text in the list (which is what LabelEncoder does)
    encoded_data = [
        float(Merchant_ID), # Merchant_ID remains a number for simplicity here
        mappings["Transaction_Type"].index(Transaction_Type),
        mappings["Payment_Gateway"].index(Payment_Gateway),
        0, # Transaction_City: Setting to 0 as the list is too long, or use a Number input
        mappings["Transaction_State"].index(Transaction_State),
        mappings["Transaction_Status"].index(Transaction_Status),
        mappings["Device_OS"].index(Device_OS),
        float(Transaction_Frequency),
        mappings["Merchant_Category"].index(Merchant_Category),
        mappings["Transaction_Channel"].index(Transaction_Channel),
        float(Transaction_Amount_Deviation),
        float(Days_Since_Last_Transaction),
        float(amount)
    ]

    data = np.array([encoded_data])

    prediction = model.predict(data)[0]
    try:
        prob = model.predict_proba(data)[0][1]
    except:
        prob = 0.5

    if prediction == 1:
        return f"<div style='color:red; font-size:22px; font-weight:bold;'>FRAUD DETECTED <br> Risk Score: {prob*100:.2f}%</div>"
    else:
        return f"<div style='color:green; font-size:22px; font-weight:bold;'>SAFE TRANSACTION <br> Risk Score: {prob*100:.2f}%</div>"

# --- UI Layout ---
css = """
/* Global */
body {
    font-family: 'Times New Roman', serif;
    background-color: #0f0f0f;
    color: white;
}

/* Main container */
.gradio-container {
    background-color: #0f0f0f !important;
}

/* Headings */
h1 {
    color: #ff6600;
    font-weight: bold;
}

h3 {
    color: #ff944d;
}

/* Cards (Input Sections) */
.gr-box {
    background: #1a1a1a !important;
    border-radius: 14px !important;
    padding: 15px !important;
    border: 1px solid #ff6600 !important;
    box-shadow: 0 4px 15px rgba(255, 102, 0, 0.3);
    transition: transform 0.2s ease, box-shadow 0.2s ease;
}

/* Hover effect */
.gr-box:hover {
    transform: translateY(-3px);
    box-shadow: 0 6px 20px rgba(255, 102, 0, 0.5);
}

/* Inputs */
input {
    background-color: #262626 !important;
    color: white !important;
    border: 1px solid #ff6600 !important;
}

/* Buttons */
button {
    background: linear-gradient(135deg, #ff6600, #ff944d) !important;
    color: white !important;
    border-radius: 10px !important;
    border: none !important;
    font-weight: bold !important;
}

/* Button hover */
button:hover {
    background: linear-gradient(135deg, #e65c00, #ff8533) !important;
}

/* Output box */
.gr-html {
    background: #1a1a1a !important;
    border: 1px solid #ff6600 !important;
    padding: 15px;
    border-radius: 12px;
    box-shadow: 0 3px 12px rgba(255, 102, 0, 0.4);
}
"""

with gr.Blocks(css=css) as demo:
    gr.Markdown("# 💳 UPI Fraud Detection System")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📌 Transaction Details")
            # Using Dropdowns instead of Numbers
            m_id = gr.Number(label="Merchant ID (Numeric)")
            t_type = gr.Dropdown(choices=mappings["Transaction_Type"], label="Transaction Type")
            p_gate = gr.Dropdown(choices=mappings["Payment_Gateway"], label="Payment Gateway")
            t_city = gr.Number(label="Transaction City")
            t_state = gr.Dropdown(choices=mappings["Transaction_State"], label="Transaction State")

        with gr.Column():
            gr.Markdown("### 📱 Device & Metrics")
            t_status = gr.Dropdown(choices=mappings["Transaction_Status"], label="Transaction Status")
            d_os = gr.Dropdown(choices=mappings["Device_OS"], label="Device OS")
            m_cat = gr.Dropdown(choices=mappings["Merchant_Category"], label="Merchant Category")
            t_chan = gr.Dropdown(choices=mappings["Transaction_Channel"], label="Transaction Channel")

        with gr.Column():
            gr.Markdown("### 💰 Financials")
            t_freq = gr.Number(label="Transaction Frequency")
            t_dev = gr.Number(label="Amount Deviation")
            t_days = gr.Number(label="Days Since Last Transaction")
            t_amt = gr.Number(label="Amount")

    output = gr.HTML()
    submit = gr.Button("🚀 Predict", variant="primary")

    submit.click(
        fn=predict,
        inputs=[m_id, t_type, p_gate, t_city, t_state, t_status, d_os, t_freq, m_cat, t_chan, t_dev, t_days, t_amt],
        outputs=output
    )

demo.launch()

/tmp/ipykernel_9704/1705721498.py:128: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://34eba40a243b72bb70.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
